# NLP: Word Embeddings и Tokenization

### План

1. Word Embeddings: Word2Vec, GloVe, FastText
2. Эмбэддинги для русского языка (navec, MTEB)
3. `nn.Embedding` в PyTorch
4. Tokenization: nltk, BPE, WordPiece, tiktoken
5. Блиц
6. Домашки

### Recap

На прошлых занятиях мы закончили блок генеративных моделей (GAN, VAE, Diffusion).
Мы уже умеем обучать нейросети (backprop, оптимизаторы, регуляризация), работать с изображениями (CNN)
и генерировать данные. Сегодня начинаем новый блок — **NLP**.

До сих пор входными данными были числа или картинки (тензоры пикселей).
Но что делать, если вход — **текст**? Нужен способ превратить слова в числовые векторы,
с которыми может работать нейросеть. Именно этим мы сегодня и займёмся.

## Word Embeddings

Нейросети работают с числами, а не с текстом. Нужен способ превратить слова в векторы.

Простейший вариант — **One-Hot Encoding**: вектор размера словаря, где 1 стоит на позиции слова.
Проблемы: огромная размерность, одинаковое расстояние между всеми словами, нет семантики.

**Word2Vec** решает эту проблему — обучает плотные векторы фиксированной размерности,
в которых семантически близкие слова оказываются рядом.

Две архитектуры Word2Vec:
- **Skip-gram** — по центральному слову предсказываем контекстные (слова вокруг). Лучше работает на маленьких корпусах и для редких слов.
- **CBOW** (Continuous Bag of Words) — по контекстным словам предсказываем центральное. Быстрее обучается на больших корпусах.

В обоих случаях обучаются **две матрицы**: эмбеддинги центральных слов и эмбеддинги контекстных слов.
После обучения используется только матрица центральных слов.

### Word2Vec (gensim)

<img src="static/w2v_skipgram.png" width=600 />

<img src="static/w2v_cbow.png" width=600 />

In [ ]:
from gensim.test.utils import common_texts
from gensim.models import Word2Vec

In [ ]:
common_texts[:10]

In [ ]:
skipgram = 1   # 1 = skip-gram, 0 = CBOW
negative_sampling = 5  # количество негативных примеров на каждый позитивный
w2v_model = Word2Vec(
    sentences=common_texts,
    vector_size=100,  # размерность эмбеддинга
    window=5,         # размер контекстного окна (сколько слов слева и справа)
    min_count=1,      # минимальная частота слова для попадания в словарь
    workers=4,
    sg=skipgram,
    negative=negative_sampling
)
w2v_model.save("word2vec.model")

In [ ]:
vector = w2v_model.wv['computer']  # get numpy vector of a word
sims = w2v_model.wv.most_similar('computer', topn=10)  # get other similar words
sims

In [ ]:
w2v_model

In [ ]:
w2v_model.wv['human'].shape

In [ ]:
result = w2v_model.most_similar(positive=['human', 'system'], negative=['computer'], topn=1)
print(result)

Skip-gram loss с negative sampling:

$$\mathcal{L} = -\log \sigma(u_{w_O}^\top v_{w_I}) - \sum_{k=1}^{K} \log \sigma(-u_{w_k}^\top v_{w_I})$$

где $v_{w_I}$ — эмбеддинг центрального слова, $u_{w_O}$ — эмбеддинг контекстного слова, $u_{w_k}$ — эмбеддинги негативных примеров.

### GloVe

**GloVe** (Global Vectors) — другой подход к обучению эмбеддингов.
В отличие от Word2Vec, который обучается на парах слов из скользящего окна,
GloVe сначала строит **матрицу со-встречаемости** $X$ по всему корпусу
($X_{ij}$ — сколько раз слово $j$ встретилось в контексте слова $i$),
а затем обучает эмбеддинги через **weighted least squares** на $\log X_{ij}$.
Редкие пары штрафуются меньше через весовую функцию.

<img src="static/glove.png" width=600 />

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip

In [ ]:
!unzip glove.6B.zip

In [ ]:
from gensim.scripts.glove2word2vec import glove2word2vec
glove2word2vec('glove.6B.100d.txt', 'glove.6B.100d.txt.word2vec')

In [ ]:
from gensim.models import KeyedVectors
# load the Stanford GloVe model
filename = 'glove.6B.100d.txt.word2vec'
glove_model = KeyedVectors.load_word2vec_format(filename, binary=False)
# calculate: (king - man) + woman = ?
result = glove_model.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)

print(result)

### [FastText](https://github.com/facebookresearch/fastText)

[python reimplementation](http://christopher5106.github.io/deep/learning/2020/04/02/fasttext_pretrained_embeddings_subword_word_representations.html)


FastText — развитие Word2Vec от Facebook. Ключевое отличие: вместо эмбеддинга целого слова
FastText представляет слово как сумму эмбеддингов его **sub-word n-грамм**.

```py
word_emb = word_emb + sum( subword_embeddings )
```

Это позволяет получать эмбеддинги для слов, которых не было в обучающей выборке (OOV - out of vocabulary).

Посмотрим, что происходит, когда Word2Vec встречает незнакомое слово:

<img src="static/fasttext.png" width=600 />

In [ ]:
w2v_model.wv['abracadabra']

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/data/cooking.stackexchange.tar.gz && tar xvzf cooking.stackexchange.tar.gz

In [ ]:
!pip install fasttext

In [ ]:
import fasttext

# Обучаем unsupervised модель (skipgram с sub-word n-граммами)
ft_model = fasttext.train_unsupervised(input="cooking.stackexchange.txt", model='skipgram', minn=3, maxn=6, dim=100)

In [ ]:
ft_model['dab']

In [ ]:
ft_model.get_subwords("abracadabra")

Примеры частых sub-word n-грамм: `ing`, `er`, `tion`, `pre`

In [ ]:
ft_model.get_word_vector("abracadabra")

### Сравнение Word2Vec, GloVe, FastText

| | Word2Vec | GloVe | FastText |
|---|---|---|---|
| 🎯 **Подход** | Предсказание контекста (skip-gram / CBOW) | Факторизация матрицы со-встречаемости | Skip-gram на sub-word n-граммах |
| 💾 **Память** | Низкая | Высокая (матрица со-встречаемости) | Средняя (n-граммы увеличивают словарь) |
| 🆕 **OOV-слова** | ❌ | ❌ | ✅ (через sub-word n-граммы) |
| 📉 **Loss** | Negative sampling | Weighted least squares на $\log X_{ij}$ | Такой же, как Word2Vec |

### Эмбэддинги для русского языка

### navec

https://github.com/natasha/navec

In [ ]:
!pip install navec

In [ ]:
!wget https://storage.yandexcloud.net/natasha-navec/packs/navec_hudlit_v1_12B_500K_300d_100q.tar

In [ ]:
from navec import Navec

path = 'navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(path)

In [ ]:
navec['вечер'].shape

### Бенчмарк эмбэддингов - [MTEB](https://arxiv.org/abs/2210.07316) [2022]

[**Leaderboard**](https://huggingface.co/spaces/mteb/leaderboard)

<img src="static/img_0.png" width=400 />

<img src="static/img_1.png" width=400 />

Мы научились использовать готовые эмбеддинги через gensim, fasttext и navec.
Теперь разберёмся, как слой эмбеддингов устроен внутри PyTorch.

## [nn.Embeddings](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

Словарь

```
1 A
2 And
3 But
4 Me
5 He
6 She

...

10000 ...
```

Токенизатор превращает текст в последовательность числовых индексов из словаря.
Например: `["Happy", "new", "year"]` → `[300, 601, 902]`.

`nn.Embedding` — это таблица, которая по индексу токена возвращает его вектор.
Размер таблицы: `(vocab_size, embedding_dim)`.

In [ ]:
# свой nn.Embeddings
import torch.nn as nn
import torch

import string

ascii_lowercase = string.ascii_lowercase

nn_embeddings = nn.Embedding(num_embeddings=len(ascii_lowercase), embedding_dim=100)

In [ ]:
nn_embeddings.weight.shape

In [ ]:
string.ascii_lowercase[0]

In [ ]:
with torch.no_grad():
    token_id = 0

    # [ 2 ]
    token_ids_t = torch.LongTensor( [ token_id, token_id ] )

    # векторное представление для буквы `a`

    # [ 2, emb_dim ] = [ 2, 100 ]
    zero_idx_embedding = nn_embeddings.forward( token_ids_t )

    # равносильно
    hands_embeddings = nn_embeddings.weight[ token_id, : ]

    assert (zero_idx_embedding == hands_embeddings).all()

zero_idx_embedding.shape

### MyEmbedding

In [ ]:
class MyEmbedding(nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()

        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim

        self.weight = nn.Parameter( torch.Tensor( size=( num_embeddings, embedding_dim ) ) )

        return

    def forward(self, token_ids): # [ bs, seq_len, ... ] # LongTensor

        token_ids_flat = token_ids.view(-1)

        new_shape = (*(token_ids.shape), self.embedding_dim)

        # [ bs, seq_len, emb_dim ]
        return self.weight[token_ids_flat, :].view( new_shape )



In [ ]:
# tokens ids
# [ 0, 1, 2, 3, ... 25 ].view(2,13).repeat(3, 1)
tokens_batch = torch.arange(26).view(2,13).repeat(3, 1)

tokens_batch.shape # [ bs, seq_len ]

In [ ]:
nn_embeddings = nn.Embedding(num_embeddings=len(ascii_lowercase), embedding_dim=100)

tokens_batch_embedded = nn_embeddings(tokens_batch)
tokens_batch_embedded.shape

In [ ]:
my_embeddings = MyEmbedding(num_embeddings=len(ascii_lowercase), embedding_dim=100)

my_tokens_batch_embedded = my_embeddings(tokens_batch)
my_tokens_batch_embedded.shape

### MyWord2Vec

In [ ]:
class PyTorchWord2Vec(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()

        self.vocab_size = vocab_size

        self.word_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.context_embedding = nn.Embedding(vocab_size, embedding_dim)

    def forward_skip_gram(self, tokens_ids, skipgram_target_tokens):
        """
        Возвращает и центральные и контекстные эмбэддинги. Используется для обучения.
        """
        # tokens_ids ~ [ seq_len ]
        # skipgram_target_tokens [ seq_len, window_size ]
        if not isinstance(tokens_ids, torch.Tensor):
            tokens_ids = torch.tensor(tokens_ids, dtype=torch.long)

        if not isinstance(skipgram_target_tokens, torch.Tensor):
            skipgram_target_tokens = torch.tensor(skipgram_target_tokens, dtype=torch.long)

        return self.word_embedding(tokens_ids), self.context_embedding(skipgram_target_tokens)

    def forward(self, tokens_ids):
        """
        Возвращает обученные эмбэддинги для центрального слова.
        Используется во время инференса. Обратите внимание, что обученная матрица
        контекстных эмбэддингов не используется.
        """
        # tokens_ids ~ [ seq_len ]
        if not isinstance(tokens_ids, torch.Tensor):
            tokens_ids = torch.tensor(tokens_ids, dtype=torch.long)

        return self.word_embedding(tokens_ids)

    def get_token_embedding(self, token_id):
        # only word_embedding used
        return self.word_embedding(token_id)

    def negative_sampling(self, num_negative_samples=20):
        """
        Возвращает негативные примеры - считаем, что у нас достаточно большой
        размер словаря, чтобы рандомные слова, вероятнее всего, были бы негативным примером
        """

        random_tokens = torch.randint(0, self.vocab_size, [num_negative_samples])

        return self.context_embedding(random_tokens)

    def build_skip_gram_list(self, tokens_ids):
        """
        Выделяет контекстные слова-токены для окна.
        """

        skip_gram_target_tokens = []

        tokens_ids = ['', *tokens_ids, '']  # source sentence padding
        for i, token_id in enumerate(tokens_ids):
            if i > 0:
                skip_gram_target_tokens.append(tokens_ids[i - 1])
            if i < len(tokens_ids) - 1:
                skip_gram_target_tokens.append(tokens_ids[i + 1])

        return skip_gram_target_tokens


In [ ]:
# Демонстрация PyTorchWord2Vec
vocab_size = 1000
embedding_dim = 50

my_w2v = PyTorchWord2Vec(vocab_size, embedding_dim)

# Прямой проход: получаем эмбеддинги центрального слова
token_ids = torch.tensor([1, 42, 7, 100])
word_embs = my_w2v(token_ids)
print(f"Входные token_ids: {token_ids}")
print(f"Выходные эмбеддинги: {word_embs.shape}")  # [4, 50]

# Skip-gram: получаем и центральные, и контекстные эмбеддинги
context_ids = torch.tensor([0, 2, 6, 8])
center_emb, context_emb = my_w2v.forward_skip_gram(token_ids, context_ids)
print(f"\nЦентральные эмбеддинги: {center_emb.shape}")
print(f"Контекстные эмбеддинги: {context_emb.shape}")

# Negative sampling
neg_embs = my_w2v.negative_sampling(num_negative_samples=10)
print(f"\nНегативные примеры: {neg_embs.shape}")  # [10, 50]

Мы разобрали, как получать векторные представления слов. Но перед тем как получить эмбеддинг,
нужно превратить текст в последовательность токенов — этим занимается **токенизация**.

## Tokenization

### nltk

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

sentence = """At eight o'clock on Thursday morning Arthur didn't feel very good. Comma, separated, sentence."""
tokens = nltk.word_tokenize(sentence)
tokens

<img src="static/img_2.png" width=400 />

In [ ]:
nltk.download('averaged_perceptron_tagger')

tagged = nltk.pos_tag(tokens)
tagged

### HuggingFace Tokenization

In [ ]:
!pip install -q transformers

### BPE Tokenizer

In [ ]:
from transformers import AutoTokenizer

# https://huggingface.co/google-bert/bert-base-uncased/blob/main/vocab.txt
# https://huggingface.co/google-bert/bert-base-uncased/blob/main/tokenizer.json
# https://huggingface.co/google-bert/bert-base-uncased/blob/main/tokenizer_config.json
# https://huggingface.co/google-bert/bert-base-uncased/blob/main/config.json

gpt2_tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')
gpt2_tokenizer

In [ ]:
tokenized = gpt2_tokenizer([ "Example tokenization" ])

[ gpt2_tokenizer._convert_id_to_token(x) for x in tokenized['input_ids'][0] ]

### Word-Piece Tokenizer

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
bert_tokenizer

In [ ]:
tokenized = bert_tokenizer([ "Example tokenization" ])

[ bert_tokenizer._convert_id_to_token(x) for x in tokenized['input_ids'][0] ]

### Сравнение BPE и WordPiece

| | BPE | WordPiece |
|---|---|---|
| **Алгоритм** | На каждом шаге объединяет самую частую пару символов/токенов | На каждом шаге объединяет пару, максимизирующую likelihood корпуса |
| **Критерий слияния** | $\arg\max \text{count}(a, b)$ | $\arg\max \frac{\text{count}(ab)}{\text{count}(a) \cdot \text{count}(b)}$ |
| **Маркер подслова** | `Ġ` перед токеном (начало нового слова) | `##` перед токеном (продолжение слова) |
| **Используется в** | GPT-2, GPT-3/4, LLaMA, и др. | BERT, DistilBERT, Electra |
| **Пример** | `tokenization` → `token`, `ization` | `tokenization` → `token`, `##ization` |

❓ **В чём разница критериев?**

BPE сливает самую частую пару — но пара `(t, h)` может быть частой
просто потому, что `t` и `h` по отдельности встречаются повсюду. Критерий WordPiece —
это по сути **pointwise mutual information**: насколько пара встречается чаще, чем ожидалось бы
при независимости. Поэтому WordPiece предпочтёт слить редкие символы, которые *всегда* ходят вместе.

Оба алгоритма обучаются **жадно** без gradient descent: на каждом шаге пересчитываются частоты токенов
в текущей сегментации, выбирается лучшая пара, добавляется в словарь, корпус пересегментируется.
Повторяется до нужного размера словаря.

❓ **Что лучше?**

Разница на downstream-задачах минимальна. Важнее размер словаря (32k/64k/128k),
данные для обучения токенизатора и покрытие языков.
[Bostrom & Durrett, 2020](https://arxiv.org/abs/2004.03720) показывают, что Unigram LM чуть лучше
для морфологически богатых языков. [Rust et al., 2021](https://arxiv.org/abs/2012.15613) показывают,
что для мультиязычных моделей важнее распределение словаря между языками, чем сам алгоритм.

❓ **Что дольше обучается?**

BPE и WordPiece имеют одинаковую сложность — $O(n)$ на каждом шаге
(подсчёт частот пар по корпусу), узкое место — пересегментация корпуса после слияния.
Unigram LM (SentencePiece) дороже: начинает с большого словаря и итеративно удаляет токены,
минимально ухудшающие likelihood. Но на практике все три обучаются за минуты-часы —
это ничто по сравнению с обучением самой LLM.

<img src="static/bpe.png" width=600 />

<img src="static/wordpiece.png" width=600 />

### TikToken

[**GitHub**](https://github.com/openai/tiktoken)



<img src="https://raw.githubusercontent.com/openai/tiktoken/main/perf.svg" width=400 />

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken

# To get the tokeniser corresponding to a specific model in the OpenAI API:

tiktoken_tokenizer = tiktoken.encoding_for_model("gpt-4o")
tiktoken_ids = tiktoken_tokenizer.encode("Example tokenization")
tiktoken_ids

In [ ]:
[tiktoken_tokenizer.decode([ x ]) for x in tiktoken_ids]

## Доп материалы

- [Voita word_embeddings](https://lena-voita.github.io/nlp_course/word_embeddings.html)
- [Habr W2V в картинках](https://habr.com/ru/post/446530/)
- [HuggingFace Course / Tokenization](https://huggingface.co/learn/nlp-course/chapter6/5)

## Блиц

#### ❓ **Вопрос**: Какие проблемы могут быть, если использовать OHE эмбэддинги для слов?

<details>

<summary><strong>Ответ</strong></summary>

* одинаковое расстояние между всеми словами, нелья как-то определить меру близости</br>
* гигантские размерности

</details>

#### ❓ **Вопрос**: Сколько матриц обучается в `Word2Vec`? А в `GloVe`?

<details>

<summary><strong>Ответ</strong></summary>

Две. Представление слова, если оно в центре. И представление, если оно контекст.

</details>

#### ❓ **Вопрос**: Какие эмбэдинги используются после обучения Word2Vec?

<details>

<summary><strong>Ответ</strong></summary>

Используются эмбэддинги центрального слова (word embedding). Контекстная матрица (context embedding) отбрасывается.</br>
Иногда используют среднее двух матриц, но на практике это редко даёт значимый прирост качества.

</details>

#### ❓ **Вопрос**: Зачем нужен negative sampling во время обучения W2V?

<details>

<summary><strong>Ответ</strong></summary>

Это оптимизация, которая позволяет обновлять фиксированное относительно небольшое количество контекстных векторов.</br>
[Voita w2v_negative_sampling](https://lena-voita.github.io/nlp_course/word_embeddings.html#w2v_negative_sampling)

</details>

#### ❓ **Вопрос**: Чем отличается Glove от Word2Vec?

<details>

<summary><strong>Ответ</strong></summary>

Обучается на другую ф-ю потерь, использует явно статистическую информацию по корпусу для штрафа редких токенов

</details>

#### ❓ **Вопрос**: Мы знаем, что Glove в отличие от Word2Vec требует больше памяти. Что именно хранится в этой памяти? Эта дополнительная память требуется во время обучения или вычисления модели? Или в обоих случаях?

<details>

<summary><strong>Ответ</strong></summary>

Доп память нужна для хранения статистик по со-встречаемости слов. Они используются только на этапе обучения для штрафа редких слов.

</details>

#### ❓ **Вопрос**: Какие преимущества у FastText по сравнению с W2V?

<details>

<summary><strong>Ответ</strong></summary>

Позволяет получить эмбеддинги для слов, которых не было в обучающей выборке за счет использования n-грамм

</details>

#### ❓ **Вопрос**: Чем лосс FastText отличается от W2V?

<details>

<summary><strong>Ответ</strong></summary>

Ничем. FastText обучается на skip-gramm лосс Word2Vec.

</details>

#### ❓ **Вопрос**: Чем `n-gram`-токенизация отличается от `BPE`-токенизации?

<details>

<summary><strong>Ответ</strong></summary>

Во-первых, метод получения эмбэддингов с помощью `n-gram` - это не токенизация, потому что в этом методе есть пересечения разных n-gram.</br>
Обычно `BPE`-токенизация дает более осмысленные куски слов. Кстати, BPE токенизацию, можно применять к Count-Based моделям, например, `FastText`

</details>

#### ❓ **Вопрос**: Какая ключевая проблема объединяет все рассмотренные методы на сегодняшнем занятии?

<details>

<summary><strong>Ответ</strong></summary>

Все эмбэддинги слов не зависят от конкретного контекста.</br>

**Например:**</br>
* Мама мыла раму</br>
* Дома нет мыла</br>
Слово "мыла" в этих двух примерах в одном случае глагол, в другом существительное. Но во всех count-based методах эмбэддинги этого слова будут одинаковыми в разных контекстах.

</details>

#### ❓ **Вопрос**: Где сейчас применяются Word2Vec, Glove, FastText?

<details>

<summary><strong>Ответ</strong></summary>

* Предобработка и фильтрация больших датасетов — скорость работы критична</br>
* Low-latency системы (поиск, рекомендации), где трансформер слишком дорог</br>
* Инициализация эмбеддинг-слоёв в лёгких моделях</br>
* Анализ и визуализация семантических отношений между словами

</details>


## Домашки

- **hw-tokenization** — нужно реализовать BPE-токенизацию.

